# CRISPR-MTL Experiments
Run each cell independently. Each experiment saves checkpoints
automatically and skips retraining if checkpoints already exist.

In [ ]:
import sys
import yaml

sys.path.insert(0, "..")
from src.train import run_experiment
from src.evaluate import generate_report, compute_integrated_gradients, plot_saliency_comparison
from src.dataset import load_ontarget, load_offtarget
from transformers import BertTokenizer
import torch
import pandas as pd

config = yaml.safe_load(open("../configs/config.yaml"))
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## Baseline: A1 — BiLSTM On-Target

In [ ]:
run_experiment("A1", config, device)

## Baseline: A2 — CNN-BiLSTM Off-Target

In [ ]:
run_experiment("A2", config, device)

## DNABERT Single-Task: B1 — On-Target

In [ ]:
run_experiment("B1", config, device)

## DNABERT Single-Task: B2 — Off-Target

In [ ]:
run_experiment("B2", config, device)

## Main Model: MTL-Full

In [ ]:
run_experiment("MTL-Full", config, device)

## Ablation: ABL1 — Frozen All Layers

In [ ]:
run_experiment("ABL1", config, device)

## Ablation: ABL2 — Unfreeze All Layers

In [ ]:
run_experiment("ABL2", config, device)

## Ablation: ABL3 — Combined Loss

In [ ]:
run_experiment("ABL3", config, device)

## Results Summary
Run after all experiments above are complete.

In [ ]:
generate_report()

## Interpretability Analysis
Run only after MTL-Full is complete.

In [ ]:
tokenizer = BertTokenizer.from_pretrained(config["model"]["dnabert_name"])
df_on  = load_ontarget(config["paths"]["ontarget_csv"])
off    = load_offtarget(config["paths"]["offtarget_dc"], config["paths"]["offtarget_lg"])
df_off = off["df"]

# Load best MTL-Full checkpoint (fold with highest combined score)
import glob
ckpt_files = sorted(glob.glob("../outputs/checkpoints/MTL-Full_fold*.pt"))
best_ckpt  = max(ckpt_files, key=lambda f: torch.load(f, map_location="cpu")["val_combined"])
print(f"Using checkpoint: {best_ckpt}")

from src.model import CRISPRMultiTask
model = CRISPRMultiTask(config)
model.load_state_dict(torch.load(best_ckpt, map_location=device)["model_state_dict"])
model = model.to(device)
model.eval()

# Select samples for IG analysis
samples_on_high = df_on.nlargest(10, "label")
samples_on_low  = df_on.nsmallest(10, "label")
samples_on      = pd.concat([samples_on_high, samples_on_low]).reset_index(drop=True)

samples_off_pos = df_off[df_off["label"] == 1].head(10)
samples_off_neg = df_off[df_off["label"] == 0].head(10)
samples_off     = pd.concat([samples_off_pos, samples_off_neg]).reset_index(drop=True)

ig_results = compute_integrated_gradients(
    model, tokenizer, samples_on, samples_off,
    n_steps=config["interpretability"]["n_steps"],
    device=device,
)
plot_saliency_comparison(ig_results, save_dir="../outputs/figures/")
print("Saliency plot saved to outputs/figures/saliency_comparison.png")